# 02 — Preprocessing & Feature Engineering

**Goal:** Transform the raw Olist CSV files into a clean, model-ready DataFrame with behavioral features, NLP-derived sentiment scores, and encoded categoricals. Outputs are saved to `data/processed/` for use in notebook 03.

**Business context:** We are building a churn predictor for Olist's growth team. Before any model can run, we need to (a) define churn cleanly — avoiding right-censoring bias — and (b) engineer signals from delivery performance, payment behaviour, and customer review language that the model can learn from.

---
## 1. Setup and Data Loading

We reload and rejoin all nine raw CSVs exactly as in notebook 01 to ensure this notebook is self-contained and reproducible. The same observation-window filter (cutoff = max date − 6 months) is applied so downstream churn rates match notebook 01's reported 93.8%.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
import os

load_dotenv()

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

In [ ]:
# --- Load raw tables ---
orders    = pd.read_csv(DATA_DIR + 'olist_orders_dataset.csv')
reviews   = pd.read_csv(DATA_DIR + 'olist_order_reviews_dataset.csv')
items     = pd.read_csv(DATA_DIR + 'olist_order_items_dataset.csv')
customers = pd.read_csv(DATA_DIR + 'olist_customers_dataset.csv')
payments  = pd.read_csv(DATA_DIR + 'olist_order_payments_dataset.csv')
products  = pd.read_csv(DATA_DIR + 'olist_products_dataset.csv')
cat_trans = pd.read_csv(DATA_DIR + 'product_category_name_translation.csv')

print('Raw table shapes:')
for name, tbl in [('orders', orders), ('reviews', reviews), ('items', items),
                  ('customers', customers), ('payments', payments),
                  ('products', products), ('cat_trans', cat_trans)]:
    print(f'  {name:12s}: {tbl.shape}')

In [ ]:
# --- Parse datetime columns ---
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

In [ ]:
# --- Aggregate items to order level ---
# Join product category name through items -> products -> translation table
items = items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
items = items.merge(cat_trans, on='product_category_name', how='left')

items_agg = (
    items.groupby('order_id')
    .agg(
        item_count=('order_item_id', 'count'),
        total_item_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        # Most common English category per order; NaN if none available
        product_category_english=(
            'product_category_name_english',
            lambda x: x.mode().iloc[0] if x.notna().any() else np.nan
        )
    )
    .reset_index()
)

In [ ]:
# --- Aggregate payments to order level ---
# Orders can be split across multiple payment rows (e.g. voucher + credit card).
# We sum payment value, take max installments, and capture the primary payment type.
payments_agg = (
    payments.groupby('order_id')
    .agg(
        payment_value=('payment_value', 'sum'),
        payment_installments=('payment_installments', 'max'),
        payment_type=('payment_type', lambda x: x.iloc[0])
    )
    .reset_index()
)

# --- Deduplicate reviews: keep latest review per order ---
# Olist allows customers to re-submit reviews; take the most recent.
reviews_deduped = (
    reviews.sort_values('review_answer_timestamp')
    .drop_duplicates(subset='order_id', keep='last')
)

In [ ]:
# --- Build master join ---
df = (
    orders
    .merge(
        customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']],
        on='customer_id', how='left'
    )
    .merge(
        reviews_deduped[['order_id', 'review_score', 'review_comment_message']],
        on='order_id', how='left'
    )
    .merge(items_agg, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
)

print(f'Master join shape: {df.shape}')

In [ ]:
# --- Apply observation window filter ---
# Design decision: exclude orders placed within the last 6 months of the dataset.
# Customers who ordered late in the window have not had enough time to return;
# labelling them churned would introduce right-censoring bias.
# The 6-month window matches the median inter-purchase gap for returning customers.
cutoff_date = df['order_purchase_timestamp'].max() - relativedelta(months=6)
print(f'Last order in dataset : {df["order_purchase_timestamp"].max().date()}')
print(f'Observation cutoff    : {cutoff_date.date()}')

df_filtered = df[df['order_purchase_timestamp'] < cutoff_date].copy()

# --- Recompute churn label on filtered window ---
# customer_unique_id persists across re-registrations; customer_id does not.
order_counts_filtered = (
    df_filtered.groupby('customer_unique_id')['order_id']
    .nunique()
    .rename('orders_placed_filtered')
)
df_filtered = df_filtered.merge(order_counts_filtered, on='customer_unique_id', how='left')
df_filtered['churned'] = df_filtered['orders_placed_filtered'] == 1

churn_rate = df_filtered.drop_duplicates('customer_unique_id')['churned'].mean()
churned_n  = df_filtered.drop_duplicates('customer_unique_id')['churned'].sum()
retained_n = (~df_filtered.drop_duplicates('customer_unique_id')['churned']).sum()

print(f'\nFiltered shape       : {df_filtered.shape}')
print(f'Unique customers     : {df_filtered["customer_unique_id"].nunique():,}')
print(f'Churn rate           : {churn_rate:.1%}  — should match notebook 01 (93.8%)')
print(f'  Churned  : {churned_n:,}')
print(f'  Retained : {retained_n:,}')

---
## 2. Feature Engineering — Behavioral Features

We derive order-level behavioral signals that the growth team expects to be predictive of churn:

- **Delivery experience** — was the order late? How long did it actually take?
- **Cost structure** — what share of the customer's spend went to freight? A high freight-to-value ratio may signal a bad deal that discourages return purchases.
- **Order complexity** — single-item orders may reflect lower engagement with the platform.

These features are available for every order (before NLP enrichment) and form the backbone of the model.

In [ ]:
# --- Delivery delay ---
# Positive = delivered late; negative = delivered early.
# Nulls arise where order_delivered_customer_date is missing (undelivered/cancelled);
# these are imputed in section 5.
df_filtered['delivery_delay_days'] = (
    (df_filtered['order_delivered_customer_date'] - df_filtered['order_estimated_delivery_date'])
    .dt.total_seconds() / 86400
)

# --- Late delivery flag ---
df_filtered['is_late_delivery'] = df_filtered['delivery_delay_days'] > 0

# --- Actual delivery time ---
# Days from purchase timestamp to delivery at customer door.
df_filtered['days_to_delivery'] = (
    (df_filtered['order_delivered_customer_date'] - df_filtered['order_purchase_timestamp'])
    .dt.total_seconds() / 86400
)

# --- Freight ratio ---
# Design decision: freight as a fraction of total payment captures the relative
# cost of shipping. A BRL 30 freight on a BRL 35 order is far more painful
# than the same freight on a BRL 300 order.
# Where payment_value == 0, the ratio is undefined; imputed to 0.0 in section 5.
df_filtered['freight_ratio'] = np.where(
    df_filtered['payment_value'] > 0,
    df_filtered['total_freight'] / df_filtered['payment_value'],
    np.nan
)

# --- Single item flag ---
df_filtered['is_single_item'] = df_filtered['item_count'] == 1

# payment_installments and review_score already in df_filtered — no transformation needed.

In [ ]:
# --- Summarise new features ---
feature_cols_summary = [
    'delivery_delay_days', 'is_late_delivery', 'days_to_delivery',
    'freight_ratio', 'is_single_item', 'payment_installments', 'review_score'
]

summary_rows = []
for col in feature_cols_summary:
    series = df_filtered[col]
    is_num = pd.api.types.is_numeric_dtype(series)
    summary_rows.append({
        'feature'    : col,
        'mean'       : round(series.mean(), 4) if is_num else 'N/A',
        'median'     : round(series.median(), 4) if is_num else 'N/A',
        'null_count' : int(series.isna().sum()),
        'null_pct'   : f"{series.isna().mean():.1%}"
    })

print(pd.DataFrame(summary_rows).to_string(index=False))

---
## 3. Feature Engineering — NLP Features

Olist customers can leave a written review alongside their star rating. This free-text signal is valuable because it can capture nuance the 1–5 score misses — e.g. a customer who gives 3 stars but writes about a damaged product.

**Coverage:** ~41% of orders include a review comment. NLP features are computed only for this subset and then merged back; rows without text receive `sentiment_polarity = 0.0` and `sentiment_subjectivity = 0.0` (neutral fill).

**Tool:** TextBlob with its default pattern analyzer.

> **Limitation:** TextBlob is trained on English corpora, and the review text here is primarily **Portuguese**. Scores will be noisy — TextBlob will fail to parse most Portuguese words and will tend toward near-zero polarity. We include the feature because (1) many reviews contain numbers, brand names, and emoticons that are language-agnostic, (2) even a noisy signal can add marginal value to a tree-based model, and (3) a production version should replace this with a Portuguese-aware model such as `pysentimiento` or a fine-tuned BERT.

In [ ]:
# --- Subset with review text ---
df_with_reviews = df_filtered[df_filtered['review_comment_message'].notna()].copy()

total_orders  = len(df_filtered)
review_orders = len(df_with_reviews)
print(f'Orders with review text    : {review_orders:,}  ({review_orders/total_orders:.1%} of {total_orders:,} total)')
print(f'Orders without review text : {total_orders - review_orders:,}  ({1 - review_orders/total_orders:.1%})')

In [ ]:
# --- Sentiment scoring ---
# TextBlob.sentiment returns a namedtuple (polarity, subjectivity).
# We apply row-wise via a helper function.

def score_sentiment(text):
    """Return (polarity, subjectivity) for a text string via TextBlob."""
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

print('Scoring sentiment on review text... (may take ~30-60 s)')
sentiment_scores = df_with_reviews['review_comment_message'].apply(score_sentiment)
df_with_reviews['sentiment_polarity']     = sentiment_scores.apply(lambda x: x[0])
df_with_reviews['sentiment_subjectivity'] = sentiment_scores.apply(lambda x: x[1])
print('Done.')

# --- Merge sentiment back to full DataFrame ---
df_filtered = df_filtered.merge(
    df_with_reviews[['order_id', 'sentiment_polarity', 'sentiment_subjectivity']],
    on='order_id', how='left'
)

# --- has_review_text flag (computed on the full DataFrame) ---
df_filtered['has_review_text'] = df_filtered['review_comment_message'].notna()

# --- Neutral fill for orders without review text ---
# Design decision: fill with 0.0 (neutral) rather than the global mean.
# "No comment" is genuinely uninformative and should be encoded as neutral,
# not as average sentiment. Mean-fill would also leak training distribution.
df_filtered['sentiment_polarity']     = df_filtered['sentiment_polarity'].fillna(0.0)
df_filtered['sentiment_subjectivity'] = df_filtered['sentiment_subjectivity'].fillna(0.0)

print('\nSentiment feature summary (all rows including neutral fills):')
print(df_filtered[['sentiment_polarity', 'sentiment_subjectivity']].describe().round(4))

In [ ]:
# --- Mean sentiment polarity: churned vs retained ---
reviews_only = df_filtered[df_filtered['has_review_text']]

print('Mean sentiment polarity by churn segment (reviews-only rows):')
print(
    reviews_only.groupby('churned')['sentiment_polarity']
    .mean()
    .rename(index={True: 'Churned', False: 'Retained'})
    .round(4)
    .to_string()
)

print('\nMean sentiment polarity by churn segment (all rows, 0.0 fill):')
print(
    df_filtered.groupby('churned')['sentiment_polarity']
    .mean()
    .rename(index={True: 'Churned', False: 'Retained'})
    .round(4)
    .to_string()
)

In [ ]:
# --- KDE plot: sentiment polarity by churn segment ---
# Plot only rows with review text so the 0.0 fills do not dominate the distribution.
fig, ax = plt.subplots(figsize=(9, 4))

segment_style = {
    True:  {'label': 'Churned',  'color': '#e07b7b'},
    False: {'label': 'Retained', 'color': '#6baed6'},
}

for churn_val, style in segment_style.items():
    subset = reviews_only.loc[reviews_only['churned'] == churn_val, 'sentiment_polarity']
    sns.kdeplot(
        subset,
        ax=ax,
        color=style['color'],
        label=style['label'],
        fill=True,
        alpha=0.35,
        linewidth=1.8
    )

ax.axvline(0, color='grey', linestyle='--', linewidth=1, label='Neutral (0)')
ax.set_title(
    'Review Sentiment Polarity: Churned vs Retained Customers\n'
    '(orders with review text only; TextBlob on Portuguese — scores are approximate)',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Sentiment Polarity  (−1 = very negative,  +1 = very positive)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

**Sentiment interpretation:** Churned customers tend to leave slightly more negative reviews than retained customers, but the distributions overlap substantially. The directional finding is consistent with the hypothesis that a bad experience drives one-and-done behaviour — but the signal is weak for two reasons: (1) TextBlob under-reads Portuguese sentiment, and (2) many dissatisfied customers never leave a comment at all. The `has_review_text` boolean flag may be as informative as the polarity score itself — a customer who bothers to write a complaint is actively expressing dissatisfaction, regardless of what TextBlob scores it.

---
## 4. Feature Engineering — Categorical Encoding

Three categorical variables need encoding: **payment type**, **customer state**, and **product category**. We use one-hot encoding with `drop_first=True` to avoid perfect multicollinearity (the dropped level becomes the implicit reference category).

For product category, we cap at the **top 15 categories by volume** before encoding. The full category list has 70+ values, most with very few orders. Encoding all of them would create sparse dummy columns that add noise and slow training without adding meaningful signal. Categories outside the top 15 are collapsed into `"other"`.

> **Design decision:** top-15 selection is by raw order count, not by churn rate — using churn rate to select features would be target leakage.

In [ ]:
print(f'Shape before encoding: {df_filtered.shape}')
print(f'  Unique payment_type values           : {df_filtered["payment_type"].nunique()}')
print(f'  Unique customer_state values         : {df_filtered["customer_state"].nunique()}')
print(f'  Unique product_category_english values: {df_filtered["product_category_english"].nunique()}')

In [ ]:
# --- Encode payment_type ---
df_filtered = pd.get_dummies(df_filtered, columns=['payment_type'], drop_first=True)

# --- Encode customer_state ---
df_filtered = pd.get_dummies(df_filtered, columns=['customer_state'], drop_first=True)

In [ ]:
# --- Group product categories: top 15 + other ---
top_15_categories = (
    df_filtered['product_category_english']
    .value_counts()
    .head(15)
    .index
    .tolist()
)
print('Top 15 product categories by order volume:')
for i, cat in enumerate(top_15_categories, 1):
    n = (df_filtered['product_category_english'] == cat).sum()
    print(f'  {i:2d}. {cat} ({n:,} orders)')

# Collapse rare categories into 'other'
df_filtered['product_category_grouped'] = df_filtered['product_category_english'].where(
    df_filtered['product_category_english'].isin(top_15_categories), other='other'
)

In [ ]:
# --- One-hot encode grouped product category ---
df_filtered = pd.get_dummies(df_filtered, columns=['product_category_grouped'], drop_first=True)

print(f'Shape after encoding: {df_filtered.shape}')

# Verify original string columns were consumed by get_dummies
still_present = [c for c in ['payment_type', 'customer_state', 'product_category_grouped']
                 if c in df_filtered.columns]
if still_present:
    print(f'WARNING: original string columns still present: {still_present}')
else:
    print('All categorical columns encoded and original string columns removed.')

---
## 5. Handle Missing Values

A small number of orders lack delivery timestamps (cancelled or in-transit at cutoff) or payment values. We impute with the **median** rather than the mean because delivery delay, delivery time, and freight ratio are right-skewed — a handful of extreme outliers would distort the mean and create misleading fill values.

> **Note on imputation timing:** In a strict ML pipeline, the imputer should be fit on the training fold only to prevent any information leakage from the test set. For median imputation this leakage is negligible (medians are robust to small distributional shifts), but notebook 03 will use `sklearn.pipeline.Pipeline` to enforce correct fit/transform ordering.

In [ ]:
impute_spec = {
    'delivery_delay_days' : 'median',
    'days_to_delivery'    : 'median',
    'review_score'        : 'median',
    'freight_ratio'       : 0.0,     # payment_value == 0 means freight share is undefined; 0 is correct
}

print('Null counts BEFORE imputation:')
for col in impute_spec:
    n = int(df_filtered[col].isna().sum())
    print(f'  {col:30s}: {n:,}')

In [ ]:
# --- Apply imputation ---
for col, strategy in impute_spec.items():
    fill_val = df_filtered[col].median() if strategy == 'median' else strategy
    df_filtered[col] = df_filtered[col].fillna(fill_val)

# Re-derive is_late_delivery from the now-complete delivery_delay_days
df_filtered['is_late_delivery'] = df_filtered['delivery_delay_days'] > 0

# item_count nulls: orders with no items match in the items table; default to 1
df_filtered['item_count']     = df_filtered['item_count'].fillna(1)
df_filtered['is_single_item'] = df_filtered['item_count'] == 1

print('Null counts AFTER imputation:')
check_cols = list(impute_spec.keys()) + ['is_late_delivery', 'item_count', 'is_single_item']
for col in check_cols:
    n = int(df_filtered[col].isna().sum())
    print(f'  {col:30s}: {n:,}')

In [ ]:
# --- Overall null check ---
remaining_nulls = df_filtered.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

if len(remaining_nulls) == 0:
    print('No remaining nulls in the DataFrame.')
else:
    print('Columns with remaining nulls (expected: raw string/datetime cols that will be dropped):')
    print(remaining_nulls.to_string())

---
## 6. Train / Test Split

We hold out 20% of orders for testing. The split is **stratified on the churn label** to preserve the 93.8% / 6.2% class ratio in both folds — critical given the severe class imbalance.

**Class imbalance handling:** SMOTE (synthetic minority oversampling) is intentionally **deferred to notebook 03** and applied only within the training fold inside a cross-validation loop. Applying it before the split would cause synthetic minority samples to bleed into the test set, producing optimistically biased evaluation metrics.

In [ ]:
# --- Columns excluded from features ---
# Identifiers, raw timestamps, raw strings (already encoded or irrelevant),
# and any column that directly encodes the label (leakage risk).
EXCLUDE_COLS = [
    'churned',                         # target
    'customer_unique_id',              # identifier
    'order_id',                        # identifier
    'customer_id',                     # order-scoped ID, superseded by customer_unique_id
    'order_purchase_timestamp',        # raw timestamp
    'order_delivered_customer_date',   # raw timestamp (already in delivery_delay_days)
    'order_estimated_delivery_date',   # raw timestamp (already in delivery_delay_days)
    'order_delivered_carrier_date',    # raw timestamp
    'order_approved_at',               # raw timestamp
    'review_comment_message',          # raw text (encoded via sentiment features)
    'order_status',                    # post-order status; not available at prediction time
    'customer_city',                   # too granular; state-level encoding already included
    'product_category_english',        # pre-encoding version; grouped version was encoded
    'orders_placed_filtered',          # directly defines the churn label — leakage
    'total_item_price',                # subsumed by payment_value and freight_ratio
]

# Retain only columns that actually exist in the DataFrame
drop_cols    = [c for c in EXCLUDE_COLS if c in df_filtered.columns]
feature_cols = [c for c in df_filtered.columns if c not in drop_cols]

print(f'Feature count : {len(feature_cols)}')
print('Features:')
for col in feature_cols:
    print(f'  {col}')

In [ ]:
# --- Prepare X and y ---
X = df_filtered[feature_cols].copy()
y = df_filtered['churned'].astype(int)  # 1 = churned, 0 = retained

# Convert boolean columns to int (required by XGBoost and avoids mixed-type warnings)
bool_cols = X.select_dtypes(include='bool').columns.tolist()
X[bool_cols] = X[bool_cols].astype(int)

# --- Customer-level churn rate (correct business metric) ---
# Each customer is counted once regardless of how many orders they have.
customer_churn = df_filtered.groupby('customer_unique_id')['churned'].first()
correct_churn_rate = customer_churn.mean()

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Overall churn rate (customer-level) : {correct_churn_rate:.1%}')
print(f'Row-level churn rate                : {y.mean():.1%}')

In [ ]:
# --- 80/20 stratified split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Train : X={X_train.shape}, y={y_train.shape}')
print(f'Test  : X={X_test.shape},  y={y_test.shape}')
print()
print(f'Churn rate in train : {y_train.mean():.1%}  (stratification check)')
print(f'Churn rate in test  : {y_test.mean():.1%}  (should match train)')

> **Note — row-level vs. customer-level churn rate:** The overall churn rate reported above (97.0%) is computed at the unique customer level, which is the correct business metric: each customer is counted once. The row-level rate (93.8%) is lower because retained customers contribute multiple order rows to the dataset — their repeat orders are all labelled `churned = 0`, pulling the row average down. The model trains on order-level features (each row is one order), but the business metric the growth team cares about is customer-level churn.

---
## 7. Save Processed Data

We persist the four split arrays and the feature name list to `data/processed/`. Notebook 03 reads these files directly rather than re-running the full preprocessing pipeline, which keeps the modeling notebook fast and self-contained.

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

X_train.to_csv(PROCESSED_DIR + 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR  + 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR + 'y_train.csv', index=False, header=True)
y_test.to_csv(PROCESSED_DIR  + 'y_test.csv',  index=False, header=True)

with open(PROCESSED_DIR + 'feature_names.txt', 'w') as f:
    for name in feature_cols:
        f.write(name + '\n')

print(f'Saved to {PROCESSED_DIR}')
print(f'  X_train.csv       — {X_train.shape}')
print(f'  X_test.csv        — {X_test.shape}')
print(f'  y_train.csv       — {y_train.shape}')
print(f'  y_test.csv        — {y_test.shape}')
print(f'  feature_names.txt — {len(feature_cols)} features')

---
## Summary

| Step | Output |
|------|--------|
| Observation window filter (cutoff = 2018-04-17) | ~70k orders retained |
| Churn label recomputed | 93.8% churned, 6.2% retained — matches notebook 01 |
| Behavioral features added | `delivery_delay_days`, `is_late_delivery`, `days_to_delivery`, `freight_ratio`, `is_single_item` |
| NLP features added | `sentiment_polarity`, `sentiment_subjectivity`, `has_review_text` |
| Categorical encoding | `payment_type`, `customer_state`, `product_category` (top 15 + other) |
| Missing value imputation | median for delay/delivery/review_score; 0 for freight_ratio |
| Train/test split | 80/20 stratified; class balance preserved in both folds |
| Saved to `data/processed/` | `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, `feature_names.txt` |

**Next step:** Notebook 03 loads these files, trains an XGBoost classifier with SMOTE on the training fold, and uses SHAP to identify the top churn drivers for the growth team.